In [ ]:
### chose to work with git bash here with following code:
### sort -R /mnt/c/pathTo/originalSentenceFile.txt | head -n 50000000 > /mnt/c/pathTo/sampleFile.txt

In [ ]:
###############################
### installing depencencies ###
###############################

# pip install wheel
# pip install -U spacy
# python -m spacy download en_core_web_sm
# pip install unix --upgrade
# pip install panda
# pip install sh

# python -m spacy download pl_core_news_md

In [ ]:
###############
### imports ###
###############

import spacy
from spacy.language import Language
from spacy_language_detection import LanguageDetector
from langdetect import detect_langs
from langdetect import detect
import unix
import pandas as pd
import re
import subprocess

In [ ]:
def get_lang_detector(nlp, name):
    return LanguageDetector(seed=42)  # We use the seed 42

In [ ]:
nlp_pl = spacy.load("pl_core_news_md")
Language.factory("language_detector", func=get_lang_detector)
nlp_pl.add_pipe('language_detector', last=True)

In [ ]:
# reading txt files in and creating a list of the sentences
# new path 'C:/Users/torto/Desktop/Studium/BA/code/subtitle_files/langSamples/pl/pl_sampleTest.txt'
# old path 'C:/Users/torto/Downloads/pl/pl_sampleTest.txt'

with open('C:/Users/torto/Desktop/Studium/BA/code/subtitle_files/langSamples/pl/pl_sampleTest.txt', 'r', encoding='utf-8') as sample:
    text = sample.read()
sentences = [s.strip() for s in text.split('\n') if s.strip()]

### now using spacy to pick out nouns
### here also using LanguageDetector to make sure it's the right language
### LanguageDetector doesn't seem to be reliable though...
for s in sentences:
    sn = nlp_pl(s)
    if sn._.language.get('language') == 'pl':
        for token in sn:
            #print(token.text, token.pos_, token.dep_)
            if token.pos_ == 'NOUN':
                continue
                #print(token.text, token.lemma_)
    elif sn._.language.get('score') >= 0.999:
        print('Sentence', sn, 'is not pl but likely',  sn._.language.get('language'))

In [ ]:
# reading txt files in and creating a list of the sentences
with open('C:/Users/torto/Desktop/Studium/BA/code/subtitle_files/langSamples/pl/pl_sampleTest.txt', 'r', encoding='utf-8') as sample:
    text = sample.read()
sentences = [s.strip() for s in text.split('\n') if s.strip()]

### now using spacy to pick out nouns
### this time using the package langdetect
for s in sentences:
    try: print(s, detect_langs(s))
    except Exception as e:
        print(e)
    sn = nlp_pl(s)

    """if sn._.language.get('language') == 'pl':
        for token in sn:
            #print(token.text, token.pos_, token.dep_)
            if token.pos_ == 'NOUN':
                continue
                #print(token.text, token.lemma_)
    elif sn._.language.get('score') >= 0.999:
        print('Sentence', sn, 'is not pl but likely',  sn._.language.get('language'))"""

In [ ]:
### reading txt files in and creating a list of the sentences
### this time NO language detection, since it will probably not matter with matching and frequency...
### here also with storage of the data in a pd data frame...

# this is the place where we would also determine the specific meaning
# maybe through a tool, maybe AI

with open('C:/Users/torto/Desktop/Studium/BA/code/subtitle_files/langSamples/pl/pl_sampleTest.txt', 'r', encoding='utf-8') as sample:
    text = sample.read()
sentences = [s.strip() for s in text.split('\n') if s.strip()]

df = pd.DataFrame(columns=['text', 'lemma', 'pos'])
### now using spacy to determine the pos tags
### this time without using LanguageDetector
for s in sentences:
    sn = nlp_pl(s)
    for token in sn:
        """if token.pos_ == 'VERB':
            print(token.text, token.lemma_, token.pos_)
        elif token.pos_ == 'NOUN':
            print(token.text, token.lemma_, token.pos_)
        elif token.pos_ == 'ADJ':
            print(token.text, token.lemma_, token.pos_)"""

### maybe we don't even need to distinguish...
### since there are forms that might be two different POS...
### we can actually already include a count ... somehow

        if token.pos_ == 'VERB' or token.pos_ == 'NOUN' or token.pos_ == 'ADJ':
            df.loc[len(df)] = [token.text, token.lemma_, token.pos_]
        else:
            continue
print(df)


In [ ]:
### reading txt files in and creating a list of the sentences
### this time NO language detection, since it will probably not matter with matching and frequency...
### here trying the option to store the data in a dictionary, for accessibility

# this is the place where we would also determine the specific meaning
# maybe through a tool, maybe AI

with open('C:/Users/torto/Desktop/Studium/BA/code/subtitle_files/langSamples/pl/pl_sampleTest.txt', 'r', encoding='utf-8') as sample:
    text = sample.read()
sentences = [s.strip() for s in text.split('\n') if s.strip()]
pl_dict = dict()
### now using spacy to determine the pos tags
### this time without using LanguageDetector
for s in sentences:
    sn = nlp_pl(s)

### is dict the way to go??? I'm starting to doubt...
    for token in sn:
        if token.pos_ == 'VERB' or token.pos_ == 'NOUN' or token.pos_ == 'ADJ':
            pl_dict[token.lemma_] = [token.text, token.pos_]
            # i think this will overwrite the same token.lemma entry...
        else:
            continue
#print(df)


In [ ]:
# function for reading in txt files of the languages from OpenSubtitles
def load_text_to_df(path, encoding='utf-8', strip_hyphens=True):
    with open(path, 'r', encoding=encoding) as file:
        text = file.read()

    if strip_hyphens:
        text = text.replace('-', '')

    # split into sentences (or paragraphs, depending on your data)
    sentences = [s.strip() for s in text.split('\n') if s.strip()]

    # create DataFrame
    df = pd.DataFrame({'sentence': sentences})
    return df

In [ ]:
def load_text_to_df_stream(in_path, samplesize, out_path, encoding='utf-8'):
    sentences = []
    with open(in_path, 'r', encoding=encoding) as file:
        for line in file:
            s = line.strip()
            if s:
                sentences.append(s)
        #sentences = random.sample(sentences, samplesize)

        with open(out_path, 'w', encoding='utf-8') as f_out:
            for s in sentences:
                f_out.write(s + '\n')

In [ ]:
load_text_to_df_stream('C:/Users/torto/Downloads/pl/pl.txt', 50000000, 'C:/Users/torto/Downloads/pl/pl_stream1.txt', encoding='utf-8')
load_text_to_df_stream('C:/Users/torto/Downloads/pl/pl.txt', 50000000, 'C:/Users/torto/Downloads/pl/pl_stream1.txt', encoding='utf-8')

In [ ]:
subprocess.run(["bash", "-c", "echo hello from bash"])

In [ ]:
# new try with sh
def shuf_sample(in_path, out_path, samplesize):
    subprocess.run(["shuf", "-n", str(samplesize), in_path, "-o", out_path], check=True)

subprocess.run(["bash", "-c", "shuf -n 10000 /mnt/c/Users/torto/Downloads/pl/pl.txt > /mnt/c/Users/torto/Downloads/pl/pl_shufSample.txt"])
#subprocess.run(["bash", "-c", "shuf --version"])

In [ ]:
shuf_sample('/mnt/c/Users/torto/Downloads/pl/pl.txt', '/mnt/c/Users/torto/Downloads/pl/pl_shufSample.txt',5)

In [ ]:
def shuf_sample2(in_path, out_path, samplesize):

    in_path = in_path.replace('\\', '/')
    out_path = out_path.replace('\\', '/')

    if in_path[1:3] == ":/":
        drive = in_path[0].lower()
        in_path_bash = f"/mnt/{drive}/{in_path[3:]}"
        out_path_bash = f"/mnt/{drive}/{out_path[3:]}"
    else:
        in_path_bash = in_path
        out_path_bash = out_path

    cmd = f"shuf -n {samplesize} '{in_path_bash}' -o '{out_path_bash}'"

    result = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)

    if result.returncode != 0:
        print("Error running shuf:")
        print(result.stderr)
    else:
        print(f"✅ Sample of {samplesize} lines saved to {out_path_bash}")

    return result.returncode


In [ ]:
shuf_sample2("C:/Users/torto/Downloads/pl/pl.txt", "C:/Users/torto/Downloads/pl/pl_shuf_T.txt", 50000000)